# 🏦 Bank Marketing - Term Deposit Subscription Prediction

**KD34403 Machine Learning for Data Science — Group 3 Project**

| NO | NAME | MATRIC NO. |
|---|---|---|
| 1. | GOO XIN YI | BI23110093 |
| 2. | TAM YI QING | BI23110092 |
| 3. | ONG CHIA YIN | BI23110241 |
| 4. | CHOK WENG HIN | BI23110182 |
| 5. | VOO XIN ROU | BI23110233 |

**Dataset:** UCI Bank Marketing (`bank-additional-full.csv`) — 41,188 rows, 20 features

**Core Question:** *Can we predict whether a bank client will subscribe to a term deposit based on their demographics and campaign interaction features?*
 

# MILESTONE 1 — DATA PIPELINE

## Step 0: Setup — Import Libraries

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print('✅ libraries imported | Random seed:', RANDOM_STATE)

## Step 1: Problem Definition

| Aspect | Details |
|---|---|
| **Business Question** | Which clients are most likely to subscribe to a term deposit after a phone campaign? |
| **ML Problem Type** | Binary Classification |
| **Input (X)** | Client demographics, financial status, past campaign interactions, macroeconomic indicators |
| **Output (y)** | Subscribed = 1 (yes) / Not subscribed = 0 (no) |
| **Success Metrics** | Balanced Accuracy, ROC-AUC, F1-Score |
| **Stakeholders** | Bank marketing teams, campaign managers |

**Importance of this study:** Helps banks identify likely subscribers, reducing marketing costs and improving campaign efficiency.

**Key Consideration:** The dataset is heavily imbalanced (~11% positive), contains sentinel values and missing entries, and includes post-call information (duration) that must be removed to prevent leakage. Proper preprocessing is critical before any model training.

## Step 2: Data Collection & Loading

In [ ]:
import os, sys, urllib.request, zipfile, shutil

DATA_FILE = 'bank-additional-full.csv'
ZIP_URL   = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'

if not os.path.exists(DATA_FILE):
    print('Downloading Bank Marketing dataset from UCI...')
    urllib.request.urlretrieve(ZIP_URL, 'bank_marketing.zip')

    # Step 1: extract inner zip from outer zip
    with zipfile.ZipFile('bank_marketing.zip', 'r') as z:
        z.extract('bank-additional.zip', '.')
    os.remove('bank_marketing.zip')

    # Step 2: find and extract CSV — internal path may vary across UCI versions
    with zipfile.ZipFile('bank-additional.zip', 'r') as z:
        names = z.namelist()
        match = next((n for n in names if DATA_FILE in n), None)
        if match is None:
            raise FileNotFoundError(
                f'{DATA_FILE} not found in archive.\n'
                f'Archive contents: {names}'
            )
        z.extract(match, '.')
        # Move to working directory root if it was extracted into a subfolder
        if match != DATA_FILE:
            shutil.move(match, DATA_FILE)
            # Clean up any empty subfolder left behind
            folder = os.path.dirname(match)
            if folder and os.path.isdir(folder):
                shutil.rmtree(folder, ignore_errors=True)

    os.remove('bank-additional.zip')
    print(f'✅ {DATA_FILE} downloaded and ready')
else:
    print(f'✅ {DATA_FILE} already present — skipping download')

In [ ]:
# Load data & Version Check
df = pd.read_csv(DATA_FILE, sep=';')

# Confirm this is bank-additional-full and not the older bank-full (45,211 rows, 17 cols)
REQUIRED_COLS = ['euribor3m', 'emp.var.rate', 'nr.employed', 'cons.price.idx', 'cons.conf.idx']
missing_cols  = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f'Wrong dataset version — missing columns: {missing_cols}')

print('✅ Correct version: bank-additional-full.csv')

print('Sample rows:')
display(df.head())

In [ ]:
# info about the dataset
print('=' * 60)
print('Dataset Info:')
print('=' * 60)

print(f'Shape: {df.shape}')

print(f'\nData Types:')
print(df.dtypes.to_string())

# True NaN values
nan_counts = df.isnull().sum()
nan_counts = nan_counts[nan_counts > 0]
print(f'\nMissing Values (NaN):')
if nan_counts.empty:
    print('  None')
else:
    print(nan_counts.to_string())

print(f'\nCategorical Feature Unique Values:')
for col in df.select_dtypes(include='object').columns:
    vals = df[col].unique()[:5]
    print(f'  {col:15s}: {list(vals)} ...')

# Summary statistics
print('=' * 60)
print('Summary Statistics — Numeric Features:')
print('=' * 60)
display(df.describe().round(2))

# Key Observations
1. **Missing Values**
There are no true missing values (NaNs) in the dataset. However, several categorical features contain the value 'unknown', which represents missing or unrecorded information.

2. **Valid Special Category**
The value 'nonexistent' in the `poutcome` feature represents clients who were not previously contacted. Unlike 'unknown', this is a meaningful category and is retained.

3. **Macroeconomic Features**
The dataset includes several macroeconomic indicators:
- `emp.var.rate` – employment variation rate (quarterly)
- `cons.price.idx` – consumer price index
- `cons.conf.idx` – consumer confidence index
- `euribor3m` – 3-month Euro Interbank Offered Rate
- `nr.employed` – number of employees (in thousands)

These variables provide economic context that may influence customer behaviour.

4. **Sentinel Value in `pdays`**
The dataset documentation indicates that a special value (−1 in some versions) represents clients who were not previously contacted. In this dataset, −1 is not present (minimum value = 0), but the value 999 appears very frequently (as observed in summary statistics).

Given its dominance and lack of realistic interpretation as a number of days, 999 is inferred to function as a sentinel value indicating no previous contact.

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw counts
counts = df['y'].value_counts()
colors = ['#FF6B6B', '#4ECDC4']
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution: Term Deposit Subscription', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Clients')
axes[0].set_xlabel('Subscribed (y)')
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 200,
                 f'{count:,}\n({count/len(df)*100:.1f}%)', ha='center', fontweight='bold')

# Right: subscription rate by job
job_rate = df.groupby('job')['y'].apply(lambda x: (x=='yes').mean()).sort_values()
axes[1].barh(job_rate.index, job_rate.values, color='#4ECDC4', edgecolor='white')
axes[1].set_title('Subscription Rate by Job Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Subscription Rate')
axes[1].axvline(x=(df['y']=='yes').mean(), color='red', linestyle='--', label='Overall rate')
axes[1].legend()
plt.tight_layout()
plt.show()

# Imbalance metrics
pos_rate  = (df['y'] == 'yes').mean()
imb_ratio = counts['no'] / counts['yes']
print(f'Positive rate: {pos_rate:.1%}  |  Imbalance ratio: {imb_ratio:.1f}:1 (no:yes)')

In [ ]:
# Numeric Feature Distributions by Target

numeric_cols = ['age', 'campaign', 'previous', 'emp.var.rate',
                'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    yes_vals = df[df['y'] == 'yes'][col]
    no_vals  = df[df['y'] == 'no'][col]
    axes[idx].hist(no_vals,  bins=30, alpha=0.6, color='#FF6B6B', label='No',  density=True, edgecolor='white')
    axes[idx].hist(yes_vals, bins=30, alpha=0.6, color='#4ECDC4', label='Yes', density=True, edgecolor='white')
    axes[idx].set_title(col, fontsize=11, fontweight='bold')
    axes[idx].legend(fontsize=8)
    axes[idx].grid(True, alpha=0.2)

fig.suptitle('Numeric Feature Distributions by Subscription Outcome', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key observation: All five macroeconomic indicators show noticeably different distributions between customers who subscribed and those who did not. this suggests that the state of the economy at the time of the call plays a big role in whether someone will say yes')

In [ ]:
#  Correlation Heatmap (Numeric Features + Target)
df_temp = df.copy()
df_temp['y_bin'] = (df_temp['y'] == 'yes').astype(int)

corr_matrix = df_temp[numeric_cols + ['y_bin']].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features vs Target (y)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation with target (y_bin), sorted by absolute value:')
print(corr_matrix['y_bin'].drop('y_bin').abs().sort_values(ascending=False).to_string())
print(" observation : nr.employed, euribor3m, and emp.var.rate are the strongest predictors "
      "but are highly intercorrelated (r up to 0.97), indicating multicollinearity.")

In [ ]:
# Categorical Feature Subscription Rates 
cat_cols_eda = ['marital', 'education', 'default', 'housing', 'loan',
                'contact', 'month', 'poutcome']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()
overall_rate = (df['y'] == 'yes').mean()

for idx, col in enumerate(cat_cols_eda):
    rate = df.groupby(col)['y'].apply(lambda x: (x=='yes').mean()).sort_values()
    rate.plot(kind='barh', ax=axes[idx], color='#4ECDC4', edgecolor='white')
    axes[idx].axvline(x=overall_rate, color='red', linestyle='--', alpha=0.7, label='Overall')
    axes[idx].set_title(f'Subscription Rate: {col}', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Rate')
    axes[idx].legend(fontsize=7)
    axes[idx].grid(True, alpha=0.2)

plt.suptitle('Subscription Rate by Categorical Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Key observation: Categorical features carry meaningful signal — "
      "poutcome, contact method, and month of contact are the strongest indicators of subscription likelihood.")

In [ ]:
# unknown Value Analysis 
unknown_counts = (df == 'unknown').sum()
unknown_counts = unknown_counts[unknown_counts > 0].sort_values()

plt.figure(figsize=(8, 5))
unknown_counts.plot(kind='barh', color='#FFA07A', edgecolor='white')
plt.title('Count of "unknown" Values per Column', fontsize=12, fontweight='bold')
plt.xlabel('Count')
plt.ylabel('Feature')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print('Columns with "unknown" values:')
print(unknown_counts.to_string())

print('It is noted that the default variable contains the largest number of "unknown" entries, which may limit its reliability as a predictor if left untreated.')

In [ ]:
#  pdays Sentinel Value Investigation
# pdays = 999 means the client was NOT previously contacted.
# Treating 999 as a numeric value would create a severely misleading feature.

print('pdays value distribution (top 10):')
print(df['pdays'].value_counts().head(10))

n_999 = (df['pdays'] == 999).sum()
print(f'\n999 sentinel: {n_999:,} rows ({n_999/len(df):.1%} of dataset)')

contacted     = df[df['pdays'] != 999]
not_contacted = df[df['pdays'] == 999]
print(f'\nSubscription rate — previously contacted:     {(contacted["y"]=="yes").mean():.1%}')
print(f'Subscription rate — NOT previously contacted: {(not_contacted["y"]=="yes").mean():.1%}')
print('\n→ Decision: replace pdays with binary flag `was_previously_contacted` and drop pdays.')
print('  This prevents the 999 sentinel from distorting any numeric model.')

In [ ]:
#  Outlier Detection (IQR method)
# Identifies outlier counts per numeric column using the 1.5×IQR rule.
# Informs whether to cap, transform, or leave outliers in preprocessing.

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'y']  # exclude target

outlier_summary = []
for col in numeric_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    outlier_summary.append({
        'Feature': col,
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Outliers': n_out,
        'Outlier %': f'{n_out/len(df)*100:.1f}%'
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)
print('Outlier Detection — IQR Method (1.5 × IQR rule)')
print('=' * 60)
display(outlier_df)

print('\nDecision: outliers are kept as-is.')
print('  - campaign outliers (many calls) reflect genuine client behaviour')
print('  - economic indicators have no true outliers — they are population-level indices')

## Step 4: Data Preprocessing 

### Preprocessing decisions 

| Decision | Justification |
|---|---|
| Drop `duration` |  This feature is only known after the call ends, so it leaks future information and cannot be used for real-world prediction.|
| Replace `pdays=999` with binary flag | The value 999 represents 'not previously contacted,' not a true numeric value. Converting it into a binary feature preserves its meaning without distorting the data. |
| Replace `'unknown'` with NaN | Treating 'unknown' as missing ensures consistent handling during imputation and avoids introducing misleading categories. |
| Ordinal encode `education` | Natural ordered hierarchy: illiterate to university.degree |
| One-hot encode other categoricals |These variables have no inherent order, so one-hot encoding prevents the model from assuming false relationships between categories. |
| Median impute numerics | Median is robust to extreme values and skewed distributions, making it suitable for real-world financial data.|
| Most-frequent impute categoricals | Replaces missing categories with the most common value, maintaining realistic distributions without introducing bias from arbitrary values. |
| Train-test split before preprocessing | Prevents data leakage by ensuring transformations are learned only from training data and then applied to unseen data.|
| StandardScaler on numerics | Ensures features are on the same scale, which is important for models like Logistic Regression and improves optimization stability.|


In [ ]:
# Initial Cleaning 
df_clean = df.copy()

# 1. Drop duration — data leakage
df_clean.drop(columns=['duration'], inplace=True)
print('✅ Dropped: duration (post-hoc leakage feature)')

# 2. Handle pdays sentinel — create binary flag, then drop pdays
df_clean['was_previously_contacted'] = (df_clean['pdays'] != 999).astype(int)
df_clean.drop(columns=['pdays'], inplace=True)
print('✅ Created: was_previously_contacted | Dropped: pdays')

# 3. Replace 'unknown' strings with NaN for proper imputation
unknown_counts = (df_clean == 'unknown').sum()
df_clean.replace('unknown', np.nan, inplace=True)
print(f'\n✅ Replaced "unknown" → NaN')
print('  Affected columns:')
print(unknown_counts[unknown_counts > 0].to_string())

# 4. Encode target
df_clean['y'] = (df_clean['y'] == 'yes').astype(int)
print(f'\n✅ Target encoded: yes=1, no=0')
print(f'  Distribution: {df_clean["y"].value_counts().to_dict()}')

In [ ]:
# Separate Features and Target
X = df_clean.drop(columns=['y'])
y = df_clean['y']

cat_cols_model = X.select_dtypes(include='object').columns.tolist()
num_cols_model = X.select_dtypes(include=np.number).columns.tolist()

print(f'Total features: {X.shape[1]}  ({len(num_cols_model)} numeric, {len(cat_cols_model)} categorical)')
print(f'\nNumeric ({len(num_cols_model)}):     {num_cols_model}')
print(f'Categorical ({len(cat_cols_model)}): {cat_cols_model}')

In [ ]:
# Stratified Train-Test Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train_raw.shape[0]:,} samples | Test: {X_test_raw.shape[0]:,} samples')
print(f'Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}')
print('✅ Stratified split preserves class ratio in both sets.')

In [ ]:
# Imputation (fit on train only)
# Numeric: median imputation (robust to outliers in age, campaign, balance)
# Categorical: most-frequent imputation (safe for job, education, marital)
num_imputer = SimpleImputer(strategy='median')
X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train_raw[num_cols_model]),
    columns=num_cols_model, index=X_train_raw.index
)
X_test_num = pd.DataFrame(
    num_imputer.transform(X_test_raw[num_cols_model]),
    columns=num_cols_model, index=X_test_raw.index
)

cat_imputer = SimpleImputer(strategy='most_frequent')
X_train_cat_imp = pd.DataFrame(
    cat_imputer.fit_transform(X_train_raw[cat_cols_model]),
    columns=cat_cols_model, index=X_train_raw.index
)
X_test_cat_imp = pd.DataFrame(
    cat_imputer.transform(X_test_raw[cat_cols_model]),
    columns=cat_cols_model, index=X_test_raw.index
)

print('✅ Numeric: median imputation (fit on train, applied to test)')
print('✅ Categorical: most-frequent imputation (fit on train, applied to test)')

In [ ]:
# Encoding
# Education has a natural ordinal hierarchy → ordinal encoding.
# All other categoricals have no inherent order → one-hot encoding.

edu_order = ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
             'high.school', 'professional.course', 'university.degree']
edu_map = {v: i for i, v in enumerate(edu_order)}  # illiterate=0, university=6

X_train_cat_enc = X_train_cat_imp.copy()
X_test_cat_enc  = X_test_cat_imp.copy()

# -1 for any remaining NaN after imputation (edge case: unseen value)
X_train_cat_enc['education'] = X_train_cat_enc['education'].map(edu_map).fillna(-1)
X_test_cat_enc['education']  = X_test_cat_enc['education'].map(edu_map).fillna(-1)

# One-hot encode the remaining categoricals
other_cat = [c for c in cat_cols_model if c != 'education']
X_train_ohe = pd.get_dummies(X_train_cat_enc, columns=other_cat, drop_first=True)
X_test_ohe  = pd.get_dummies(X_test_cat_enc,  columns=other_cat, drop_first=True)

# Align columns — test set may be missing dummy columns if a category was rare
X_test_ohe = X_test_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

print(f'✅ Ordinal: education ({len(edu_order)} levels, 0=illiterate → 6=university)')
print(f'✅ One-hot: {other_cat}')
print(f'   Categorical block expanded: → {X_train_ohe.shape[1]} columns')

In [ ]:
# Assemble Baseline Feature Matrix & Scale
# Reset indices to avoid mismatch
X_train_num = X_train_num.reset_index(drop=True)
X_test_num  = X_test_num.reset_index(drop=True)
X_train_ohe = X_train_ohe.reset_index(drop=True)
X_test_ohe  = X_test_ohe.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

# Identify continuous numeric columns (exclude binary flags)
binary_cols = ['was_previously_contacted']  
continuous_cols = [col for col in X_train_num.columns if col not in binary_cols]

# Initialize scaler
scaler = StandardScaler()

# Scale only continuous features
X_train_num_scaled = X_train_num.copy()
X_test_num_scaled  = X_test_num.copy()

X_train_num_scaled[continuous_cols] = scaler.fit_transform(X_train_num[continuous_cols])
X_test_num_scaled[continuous_cols]  = scaler.transform(X_test_num[continuous_cols])

# Combine scaled numeric features with one-hot encoded categorical features
X_train_scaled = pd.concat([X_train_num_scaled, X_train_ohe], axis=1)
X_test_scaled  = pd.concat([X_test_num_scaled, X_test_ohe], axis=1)

baseline_feature_names = X_train_scaled.columns.tolist()

print(f'✅ Baseline feature matrix assembled and scaled')
print(f'   Train: {X_train_scaled.shape}  |  Test: {X_test_scaled.shape}')
print(f'Sanity check — NaN in train: {X_train_scaled.isnull().sum().sum()}')
print(f'Sanity check — NaN in test:  {X_test_scaled.isnull().sum().sum()}')

print('\nExample scaled features (first 3 samples):')
display(X_train_scaled.head(3).round(2))

## Baseline Model: Logistic Regression
- only client demographics is used for training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# Select only demographic + campaign features (exclude macroeconomic indicators)
macro_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

X_train_baseline = X_train_scaled.drop(columns=[c for c in macro_cols if c in X_train_scaled.columns])
X_test_baseline  = X_test_scaled.drop(columns=[c for c in macro_cols if c in X_test_scaled.columns])

print(f'✅ Baseline feature set: {X_train_baseline.shape[1]} features (demographic + campaign only)')
print(f'   Dropped macro cols : {macro_cols}')

lr_baseline = LogisticRegression(
    solver='saga',
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print('\n✅ Baseline Logistic Regression architecture defined')
print(f'   Solver       : saga')
print(f'   Penalty      : L2')
print(f'   C            : 1.0  (default, no tuning)')
print(f'   class_weight : Balanced')

# Train
lr_baseline.fit(X_train_baseline, y_train)
print('✅ Baseline model trained')

# Predict
y_pred_baseline       = lr_baseline.predict(X_test_baseline)
y_pred_proba_baseline = lr_baseline.predict_proba(X_test_baseline)[:, 1]

# Metrics
accuracy_baseline     = accuracy_score(y_test, y_pred_baseline)
balanced_acc_baseline = balanced_accuracy_score(y_test, y_pred_baseline)
f1_baseline           = f1_score(y_test, y_pred_baseline)
roc_auc_baseline      = roc_auc_score(y_test, y_pred_proba_baseline)

print('\nBASELINE MODEL PERFORMANCE')
print('=' * 60)
print(f'Accuracy:          {accuracy_baseline:.1%}')
print(f'Balanced Accuracy: {balanced_acc_baseline:.1%}')
print(f'F1-Score:          {f1_baseline:.1%}')
print(f'ROC-AUC:           {roc_auc_baseline:.3f}')

# Plots
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred No', 'Pred Yes'],
            yticklabels=['Actually No', 'Actually Yes'])
axes[0].set_title('Confusion Matrix (Baseline)', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_pred_proba_baseline)
axes[1].plot(fpr, tpr, color='#4ECDC4', lw=3, label=f'Baseline (AUC={roc_auc_baseline:.3f})')
axes[1].plot([0, 1], [0, 1], '--', color='#FF6B6B', label='Random')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_baseline, target_names=['No (0)', 'Yes (1)']))

## Feature Engineering

**Three goals**:
1. Fix multicollinearity — PCA on 5 correlated economic cols
2. Two interaction terms — poutcome×previous, campaign×previous
3. Age binning — 4 groups replacing raw age

_*All fit objects (PCA, scaler) fitted on training data only._

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Start from raw split, bring in ALL columns now
df_tr = X_train_raw.copy()
df_te = X_test_raw.copy()

# Re-impute (fit on train only)
all_num = X_train_raw.select_dtypes(include=np.number).columns.tolist()
all_cat = X_train_raw.select_dtypes(include='object').columns.tolist()

num_imp_eng = SimpleImputer(strategy='median')
df_tr[all_num] = num_imp_eng.fit_transform(df_tr[all_num])
df_te[all_num] = num_imp_eng.transform(df_te[all_num])

cat_imp_eng = SimpleImputer(strategy='most_frequent')
df_tr[all_cat] = cat_imp_eng.fit_transform(df_tr[all_cat])
df_te[all_cat] = cat_imp_eng.transform(df_te[all_cat])

# =======================================
# Goal 1: Fix multicollinearity via PCA
# =======================================

# Group 1: labour-market block (r=0.97)
econ_block = ['emp.var.rate', 'euribor3m', 'nr.employed']

# Group 2: consumer sentiment block
cons_block  = ['cons.price.idx', 'cons.conf.idx']

# Scale each block before PCA (PCA is sensitive to scale)
scaler_econ = StandardScaler()
scaler_cons = StandardScaler()

econ_tr_scaled = scaler_econ.fit_transform(df_tr[econ_block])
econ_te_scaled = scaler_econ.transform(df_te[econ_block])

cons_tr_scaled = scaler_cons.fit_transform(df_tr[cons_block])
cons_te_scaled = scaler_cons.transform(df_te[cons_block])

# PCA → 1 component each (captures >95% variance given the high correlation)
pca_econ = PCA(n_components=1, random_state=RANDOM_STATE)
pca_cons = PCA(n_components=1, random_state=RANDOM_STATE)

df_tr['econ_index'] = pca_econ.fit_transform(econ_tr_scaled)
df_te['econ_index'] = pca_econ.transform(econ_te_scaled)

df_tr['cons_index'] = pca_cons.fit_transform(cons_tr_scaled)
df_te['cons_index'] = pca_cons.transform(cons_te_scaled)

print(f'econ_index explains {pca_econ.explained_variance_ratio_[0]:.1%} of variance in {econ_block}')
print(f'cons_index explains {pca_cons.explained_variance_ratio_[0]:.1%} of variance in {cons_block}')

# =====================================
# Goal 2: Two interaction terms
# =====================================

# Interaction 1: previous campaign success × was previously contacted
# poutcome = 'success' is the strongest single categorical signal
df_tr['prev_success'] = (df_tr['poutcome'] == 'success').astype(int)
df_te['prev_success'] = (df_te['poutcome'] == 'success').astype(int)

df_tr['poutcome_x_contact'] = df_tr['prev_success'] * df_tr['was_previously_contacted']
df_te['poutcome_x_contact'] = df_te['prev_success'] * df_te['was_previously_contacted']

# Interaction 2: campaign effort × previous contacts
# High campaign calls + low previous = first-time aggressive outreach
# Low campaign calls + high previous = warm re-engagement
df_tr['campaign_x_previous'] = df_tr['campaign'] * (df_tr['previous'] + 1)
df_te['campaign_x_previous'] = df_te['campaign'] * (df_te['previous'] + 1)

# =====================================================
# Goal 3: Age binning (nonlinear subscription pattern)
# =====================================================
# young (<30): less financially stable, low subscription
# middle (30–45): busy with mortgages/family, moderate
# senior (45–60): peak savings period, higher subscription
# retired (60+): conservative investors, high subscription

bins   = [0, 30, 45, 60, 100]
labels = [0, 1, 2, 3]   # ordinal: young=0, retired=3

df_tr['age_group'] = pd.cut(df_tr['age'], bins=bins, labels=labels, right=False).astype(int)
df_te['age_group'] = pd.cut(df_te['age'], bins=bins, labels=labels, right=False).astype(int)

# Assemble enhanced feature matrix

# Numeric: baseline cols (drop raw age, replace with age_group)
#          + engineered numerics
#          + drop the 5 raw economic cols (replaced by PCA components)

eng_num_cols = [
    'age_group', 'campaign', 'previous', 'was_previously_contacted',
    'econ_index', 'cons_index',
    'prev_success', 'poutcome_x_contact', 'campaign_x_previous'
]

# Categorical: same as baseline (education ordinal, rest one-hot)
eng_cat_cols = [
    'job', 'marital', 'education',
    'default', 'housing', 'loan',
    'contact', 'month', 'day_of_week',
    'poutcome'
]

df_tr['education'] = df_tr['education'].map(edu_map).fillna(-1)
df_te['education'] = df_te['education'].map(edu_map).fillna(-1)

ohe_cols_eng = [c for c in eng_cat_cols if c != 'education']
df_tr_ohe = pd.get_dummies(df_tr[ohe_cols_eng], drop_first=True)
df_te_ohe  = pd.get_dummies(df_te[ohe_cols_eng],  drop_first=True)
df_te_ohe  = df_te_ohe.reindex(columns=df_tr_ohe.columns, fill_value=0)

X_eng_tr = pd.concat([
    df_tr[eng_num_cols + ['education']].reset_index(drop=True),
    df_tr_ohe.reset_index(drop=True)
], axis=1)

X_eng_te = pd.concat([
    df_te[eng_num_cols + ['education']].reset_index(drop=True),
    df_te_ohe.reset_index(drop=True)
], axis=1)

# Scale to fit on train only
binary_eng  = ['was_previously_contacted', 'prev_success', 'poutcome_x_contact']
scale_eng   = [c for c in eng_num_cols if c not in binary_eng]

scaler_eng = StandardScaler()
X_eng_tr[scale_eng] = scaler_eng.fit_transform(X_eng_tr[scale_eng])
X_eng_te[scale_eng] = scaler_eng.transform(X_eng_te[scale_eng])

enhanced_feature_names = X_eng_tr.columns.tolist()

print(f'\nBaseline features : {X_bl_tr.shape[1]}')
print(f'Enhanced features : {X_eng_tr.shape[1]}')
print(f'NaN in train: {X_eng_tr.isnull().sum().sum()} | NaN in test: {X_eng_te.isnull().sum().sum()}')

In [ ]:
# milestone 4: Model Optimization


In [ ]:
# milestone 5: Final Evaluation 
